In [7]:
import os
import pathlib
import timeit
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, recall_score

In [2]:
TFLITE_DIR = pathlib.Path("../models/tflite")
NOTEBOOK_DIR = pathlib.Path(".")

mlp_tflite_models = {
    "MLP_Original_baseline":   ("MLP_Original_baseline.tflite", False),
    "MLP_Original_quant":      ("MLP_Original_quant.tflite", True),
    "MLP_Node_baseline":       ("MLP_Node_baseline.tflite", False),
    "MLP_Node_quant":          ("MLP_Node_quant.tflite", True),
    "MLP_Weight_baseline":     ("MLP_Weight_baseline.tflite", False),
    "MLP_Weight_quant":        ("MLP_Weight_quant.tflite", True),
    "MLP_WeightNode_baseline": ("MLP_WeightNode_baseline.tflite", False),
    "MLP_WeightNode_quant":    ("MLP_WeightNode_quant.tflite", True),
}

print("Models found:")
for name, (filename, _) in mlp_tflite_models.items():
    path = TFLITE_DIR / filename
    print(f"  {name}: {'OK' if path.exists() else 'NOT FOUND'} — {path}")

Models found:
  MLP_Original_baseline: OK — ..\models\tflite\MLP_Original_baseline.tflite
  MLP_Original_quant: OK — ..\models\tflite\MLP_Original_quant.tflite
  MLP_Node_baseline: OK — ..\models\tflite\MLP_Node_baseline.tflite
  MLP_Node_quant: OK — ..\models\tflite\MLP_Node_quant.tflite
  MLP_Weight_baseline: OK — ..\models\tflite\MLP_Weight_baseline.tflite
  MLP_Weight_quant: OK — ..\models\tflite\MLP_Weight_quant.tflite
  MLP_WeightNode_baseline: OK — ..\models\tflite\MLP_WeightNode_baseline.tflite
  MLP_WeightNode_quant: OK — ..\models\tflite\MLP_WeightNode_quant.tflite


In [3]:
data = np.load('../data/fdia_dataset_processed.npz')
X_test = data['X_test']       # (N, 6, 83) — mantém assim, sem achatar
y_test = data['y_test']

print(f"X_test: {X_test.shape}")

X_test: (9720, 6, 83)


In [8]:
def measure_batch_inference_tflite(tflite_path, X_test, quantized,
                                    timing_fraction=0.25, n_repeats=5, seed=42):
    interpreter = tf.lite.Interpreter(
        model_path=str(tflite_path),
        num_threads=1,
        experimental_delegates=[]
    )
    interpreter.allocate_tensors()
    in_d = interpreter.get_input_details()[0]
    out_d = interpreter.get_output_details()[0]

    if quantized:
        input_scale, input_zero_point = in_d['quantization']

    rng = np.random.default_rng(seed)
    sample_size = int(timing_fraction * len(X_test))

    batch_times = []

    for _ in range(n_repeats):
        idx = rng.choice(len(X_test), size=sample_size, replace=False)
        X_batch = X_test[idx].astype(np.float32)

        if quantized:
            X_batch = (X_batch / input_scale + input_zero_point).astype(np.uint8)

        start = timeit.default_timer()
        for sample in X_batch:
            interpreter.set_tensor(in_d['index'], np.expand_dims(sample, axis=0))
            interpreter.invoke()
            _ = interpreter.get_tensor(out_d['index'])
        end = timeit.default_timer()

        elapsed_ms = (end - start) * 1000
        per_sample_ms = elapsed_ms / sample_size
        batch_times.append(per_sample_ms)

    del interpreter
    return np.mean(batch_times), batch_times

def evaluate_tflite(tflite_path, X_test, y_test, quantized):
    interpreter = tf.lite.Interpreter(
        model_path=str(tflite_path),
        num_threads=1,
        experimental_delegates=[]
    )
    interpreter.allocate_tensors()

    in_d = interpreter.get_input_details()[0]
    out_d = interpreter.get_output_details()[0]

    if quantized:
        input_scale, input_zero_point = in_d["quantization"]
        output_scale, output_zero_point = out_d["quantization"]

    predictions = []

    for sample in X_test:
        x = np.expand_dims(sample.astype(np.float32), axis=0)

        if quantized:
            x = (x / input_scale + input_zero_point).astype(np.uint8)

        interpreter.set_tensor(in_d["index"], x)
        interpreter.invoke()

        output = interpreter.get_tensor(out_d["index"])

        if quantized and output_scale > 0:
            output = (
                output.astype(np.float32) - output_zero_point
            ) * output_scale

        predictions.append(float(output.squeeze()))

    y_pred = (np.array(predictions) >= 0.5).astype(int)

    del interpreter

    return (
        accuracy_score(y_test, y_pred),
        recall_score(y_test, y_pred, pos_label=1),
        recall_score(y_test, y_pred, pos_label=0)
    )

In [9]:
results = []

for name, (filename, quantized) in mlp_tflite_models.items():
    path = TFLITE_DIR / filename

    # Timing
    avg_time, all_times = measure_batch_inference_tflite(
        path,
        X_test,
        quantized,
        timing_fraction=0.25,
        n_repeats=100
    )

    # Metrics
    accuracy, fdia_recall, fault_recall = evaluate_tflite(
        path,
        X_test,
        y_test,
        quantized
    )

    # Model file size
    model_size_kb = path.stat().st_size / 1024

    print(f"\n{name}")
    print(f"Accuracy:       {accuracy:.6f}")
    print(f"FDIA Recall:    {fdia_recall:.6f}")
    print(f"Fault Recall:   {fault_recall:.6f}")
    print(f"Model Size:     {model_size_kb:.3f} KB")
    print(f"Inference Time: {avg_time:.6f} ms/sample")

    results.append({
        "architecture": "MLP",
        "model": name,
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "model_size_kb": model_size_kb,
        "inference_time_ms": avg_time
    })

df_batch_tflite = pd.DataFrame(results)

df_batch_tflite.to_csv(
    NOTEBOOK_DIR / "MLP_tflite_batch_inference_results.csv",
    index=False
)

df_batch_tflite

c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



MLP_Original_baseline
Accuracy:       0.998354
FDIA Recall:    0.998264
Fault Recall:   0.998435
Model Size:     179.320 KB
Inference Time: 0.005603 ms/sample


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



MLP_Original_quant
Accuracy:       0.996708
FDIA Recall:    0.996528
Fault Recall:   0.996870
Model Size:     57.664 KB
Inference Time: 0.004795 ms/sample


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


...

c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



MLP_WeightNode_baseline
Accuracy:       0.986214
FDIA Recall:    0.971354
Fault Recall:   0.999609
Model Size:     114.676 KB
Inference Time: 0.005670 ms/sample


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



MLP_WeightNode_quant
Accuracy:       0.986934
FDIA Recall:    0.974175
Fault Recall:   0.998435
Model Size:     39.594 KB
Inference Time: 0.004742 ms/sample


,architecture,model,accuracy,fdia_recall,fault_recall,model_size_kb,inference_time_ms
0,MLP,MLP_Original_baseline,0.998354,0.998264,0.998435,179.320312,0.005603
1,MLP,MLP_Original_quant,0.996708,0.996528,0.996870,57.664062,0.004795
2,MLP,MLP_Node_baseline,0.995988,0.994141,0.997653,116.609375,0.004694
3,MLP,MLP_Node_quant,0.993724,0.990885,0.996283,39.320312,0.004549
4,MLP,MLP_Weight_baseline,0.998148,0.998264,0.998044,179.246094,0.005489
5,MLP,MLP_Weight_quant,0.997016,0.997613,0.996479,57.585938,0.004825
6,MLP,MLP_WeightNode_baseline,0.986214,0.971354,0.999609,114.675781,0.005670
7,MLP,MLP_WeightNode_quant,0.986934,0.974175,0.998435,39.593750,0.004742
